## 1. Import Libraries

In [ ]:
import os
import sys
import yaml
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("✓ All imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load Configuration

## 3. Set All Random Seeds for Reproducibility

In [ ]:
import random

def set_all_seeds(seed=42, deterministic=True):
    """
    Set all random seeds for reproducibility.
    This ensures EXACT same results across runs.
    """
    print(f"Setting all random seeds to {seed}...")
    
    # Python random
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # NumPy
    np.random.seed(seed)
    
    # PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # cuDNN
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        print("  ✓ Deterministic mode enabled (slower but reproducible)")
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True
        print("  ✓ Benchmark mode enabled (faster but non-deterministic)")
    
    print("✓ All random seeds set successfully!")

# Set seeds from config
set_all_seeds(
    seed=config['reproducibility']['seed'],
    deterministic=config['reproducibility']['deterministic']
)

## 4. Verify Directory Structure

In [ ]:
print("Checking directory structure...\n")

# Check raw data
raw_data_dir = Path(config['paths']['raw_data'])
print(f"Raw Data Directory: {raw_data_dir}")
if raw_data_dir.exists():
    print("  ✓ Training folder:", (raw_data_dir / 'Training').exists())
    print("  ✓ Testing folder:", (raw_data_dir / 'Testing').exists())
else:
    print("  ✗ Raw data directory not found!")

# Check preprocessed data
preprocessed_dir = Path(config['paths']['preprocessed_data'])
print(f"\nPreprocessed Data Directory: {preprocessed_dir}")
if preprocessed_dir.exists():
    for split in ['train', 'val', 'internal_test', 'heldout_test']:
        split_dir = preprocessed_dir / split
        if split_dir.exists():
            # Count images
            num_images = sum(1 for _ in split_dir.rglob('*.jpg')) + sum(1 for _ in split_dir.rglob('*.png'))
            print(f"  ✓ {split}/: {num_images} images")
        else:
            print(f"  ✗ {split}/ not found")
else:
    print("  ✗ Preprocessed data not found!")
    print("  → Run preprocessing_FIXED.ipynb first!")

# Create output directories
for dir_name in ['checkpoints', 'results', 'logs', 'figures']:
    dir_path = Path(config['paths'][dir_name])
    dir_path.mkdir(parents=True, exist_ok=True)
    print(f"\n✓ {dir_name}/ directory ready")

## 5. Device Configuration

In [ ]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {device}")

if device.type == 'cuda':
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("⚠️  No GPU detected - training will be slow!")
    print("   Consider using Google Colab or a cloud GPU instance.")

## 6. Export Key Variables for Other Notebooks

In [ ]:
# Store these for use in other notebooks
%store config
%store device

print("\n✓ Setup complete!")
print("\nNext steps:")
print("  1. If preprocessed data not found → Run: preprocessing_FIXED.ipynb")
print("  2. To train main model → Run: 02_train_tumornet_lite.ipynb")
print("  3. For ablation study → Run: 03_ablation_study.ipynb")
print("  4. For baseline comparison → Run: 04_baseline_comparison.ipynb")